# testing

> The doubles: a host with no disk, backends with no model, and a script that behaves like a bad local engine.

Every failure the harness is written to survive can be reproduced here without downloading
a model or touching a filesystem. These are library code rather than test fixtures because
three different suites and the docs pages all need them, and because a double that only
exists inside `tests/` cannot be used to demonstrate anything on a documentation page.

In [ ]:
#| default_exp testing

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail
from ramabana.tools import file_tools, memory_tools, session_tools, tools_for, web_tools

In [ ]:
#| export
import time
from pathlib import Path
from ramabana.core import AgentError, ModelSpec, agent_err, register_model
from ramabana.runtime import Backend, Usage
from ramabana.tools import AttrDict, Hit, LocalHost, NullHost
from ramabana.agent import Agent

## Specs

Two specs, and the window sizes are the whole point. `SPEC` is small because compaction
arithmetic is easier to check against 1000 tokens than against 200,000; `SCRIPTED` is big
because a screenshot must not have compaction firing across it.

In [ ]:
#| export
SPEC = ModelSpec('fake', 'fake', 'fake/model', ctx=1000)

#: `ScriptedBackend`'s own spec, kept distinct because it is used for screenshots rather
#: than for tests -- a capture of the IDE should not have "fake" written across the status
#: bar, and its window has to be big enough that compaction does not fire mid-screenshot.
SCRIPTED = ModelSpec('scripted', 'scripted', 'scripted/model', ctx=8000)

In [ ]:
SPEC, SCRIPTED.ctx

(ModelSpec(name='fake', backend='fake', model_id='fake/model', ctx=1000, note='', config={}),
 8000)

## A host in memory

`MemHost` puts the open folders in a dict, so every file and search tool can be driven
without touching disk. It inherits `NullHost`, which means everything it does *not* define
is still absent -- and the tool list shrinks accordingly, which is the behaviour worth
testing.

In [ ]:
#| export
class MemHost(NullHost):
    "A host whose folders live in a dict, so the file tools can be driven without touching disk."

    def __init__(self, files=None, root='/proj', commands=None):
        super().__init__([root])
        self.files, self.root = dict(files or {}), root
        self.ran, self.cmds = [], []
        # What a command is scripted to return: `{command: (exit_code, output)}`. Anything
        # not scripted exits 0 with no output, so a test that only cares *that* a command
        # ran does not have to describe one.
        self.commands = dict(commands or {})

    def run_cmd(self, command, cwd=None, timeout=120):
        if not str(command or '').strip(): return 0, ''      # the capability probe
        self.cmds.append((command, cwd, timeout))
        return self.commands.get(command, (0, ''))

    @property
    def shell_note(self): return 'scripted'

    def check(self, path, must_exist=False):
        p = Path(path)
        return p if p.is_absolute() else Path(self.root)/p

    def walk(self): return list(self.files)
    def read(self, path): return self.files.get(str(path))
    def text_at(self, path): return self.files.get(str(path), '')

    def write(self, path, text):
        self.files[str(path)] = text
        return str(path)

    def search(self, query, limit=20):
        return [Hit(p, 1, '', t.splitlines()[0]) for p, t in self.files.items() if query in t][:limit]

    @property
    def search_note(self): return 'memory'

    def run_python(self, code):
        self.ran.append(code)
        return 'ok'

In [ ]:
host = MemHost({'/proj/a.py': 'def a(): return 1\n', '/proj/b.py': 'def b(): return a() + 1\n'})
host.roots, host.walk()

(['/proj'], ['/proj/a.py', '/proj/b.py'])

Search is a substring scan that returns real `Hit`s, so a caller cannot tell the shape of
these results from an index's.

In [ ]:
host.search('return a'), host.search_note

([/proj/b.py:1    def b(): return a() + 1], 'memory')

Writes land in the dict and are visible to `text_at`, which is what makes `Agent.changes()`
testable: the before/after comparison is against a real (if imaginary) file.

In [ ]:
host.write('/proj/c.py', 'def c(): pass\n')
host.text_at('/proj/c.py'), host.text_at('/proj/never.py')

('def c(): pass\n', '')

`run_python` records instead of executing, so a test can assert what the agent tried to run
in the user's namespace without a kernel.

In [ ]:
host.run_python('df2 = df.dropna()'), host.ran

('ok', ['df2 = df.dropna()'])

In [ ]:
test_fail(lambda: host.list_vars(), NotImplementedError)      # still absent: no session tools
host.kernel_kind

'ipykernel'

## A host with everything

`LocalHost` is honest about what a terminal cannot do -- no research memory, no IDE
terminal, no overlay scope -- so `tools_for` drops those tools, which makes it a poor place
to test the tools it drops. `FullHost` fills each gap with a small real implementation over
in-process state: a remembered-pages index that genuinely searches, and a web that genuinely
returns the pages it was handed. Nothing reaches the network, and the folder is a fresh
temporary one, so a test or a docs page can write freely.

In [ ]:
#| export
class FullHost(LocalHost):
    """A host with *every* capability present, over a throwaway folder.

    `LocalHost` is honest about what a terminal cannot do: it has no research memory, no
    IDE terminal and no overlay scope, so `tools_for` drops those tools. That makes it a
    poor place to test the tools it drops. `FullHost` fills each gap with a small real
    implementation over in-process state -- a remembered-pages index that genuinely
    searches, a web that genuinely returns the pages it was given -- so the whole tool
    surface can be driven, and every group's contract has somewhere to be checked.

    Nothing here reaches the network, and the folder is a fresh temporary one unless a root
    is given, so a test can write freely.
    """

    def __init__(self, files=None, pages=None, root=None, terminal='', **kw):
        import tempfile
        root = Path(root) if root else Path(tempfile.mkdtemp())/'proj'
        root.mkdir(parents=True, exist_ok=True)
        super().__init__([root], web=False, index=False, **kw)
        self.root = self.check('.')
        for path, text in (files or {}).items(): self.write(path, text)
        #: url -> markdown, the entire web this host knows about
        self.pages = dict(pages or {})
        self.transcript = [x for x in str(terminal).splitlines() if x]
        self.remembered = {}      # doc_id -> (title, url, markdown)
        self.notes = []           # everything `note` was told, for a test to assert on

    # -- the web, without a network -----------------------------------------
    def web_search(self, query, n=20):
        q = str(query).lower()
        hits = [(url, text) for url, text in self.pages.items()
                if q in url.lower() or q in text.lower()]
        return [AttrDict(title=text.strip().splitlines()[0].lstrip('# ').strip(), url=url)
                for url, text in hits[:int(n)]]

    def read_url(self, url, remember=True):
        text = self.pages.get(str(url))
        if text is None: return None
        if remember: self._remember_page(str(url), text)
        return AttrDict(text=text, url=str(url))

    def research(self, query):
        "Read every page that matches, into one cited digest -- what a real research pass returns."
        hits = self.web_search(query)
        if not hits: return ''
        parts = []
        for h in hits:
            page = self.read_url(h.url)
            parts.append(f'## {h.title}\n\n{page.text.strip()}\n\n[source: {h.url}]')
        return f'# {query}\n\n' + '\n\n'.join(parts)

    @property
    def research_note(self): return f'{len(self.pages)} page(s) in this host'

    # -- research memory, as a real tree ------------------------------------
    def _sections(self, text):
        "A markdown document split at its headings, which is the unit memory recalls."
        out, head, body = [], '', []
        for line in str(text).splitlines():
            if line.startswith('#'):
                if head or body: out.append((head, '\n'.join(body).strip()))
                head, body = line.lstrip('# ').strip(), []
            else: body.append(line)
        if head or body: out.append((head, '\n'.join(body).strip()))
        return [(h, b) for h, b in out if h or b]

    def _remember_page(self, url, text):
        doc_id = f'doc_{len(self.remembered) + 1}'
        title = self._sections(text)[0][0] if self._sections(text) else url
        self.remembered[doc_id] = (title or url, url, str(text))
        return doc_id

    def memory_search(self, query, limit=8):
        q = str(query).lower()
        rows = []
        for doc_id, (title, url, text) in self.remembered.items():
            for i, (head, body) in enumerate(self._sections(text)):
                if q not in f'{head}\n{body}'.lower(): continue
                crumb = title if (not head or head == title) else f'{title} › {head}'
                rows.append({'node_id': f'{doc_id}#{i}', 'document': title, 'url': url,
                             'breadcrumb': crumb, 'text': body})
        return rows[:int(limit)]

    def memory_tree(self, document=''):
        d = str(document).lower()
        return [{'doc_id': doc_id, 'title': title, 'url': url,
                 'headings': [{'node_id': f'{doc_id}#{i}', 'heading': head}
                              for i, (head, _) in enumerate(self._sections(text))]}
                for doc_id, (title, url, text) in self.remembered.items()
                if not d or d in title.lower() or d == doc_id]

    def memory_read(self, node_id):
        doc_id, _, idx = str(node_id).partition('#')
        if doc_id not in self.remembered: raise AgentError(f'no remembered node {node_id!r}')
        title, url, text = self.remembered[doc_id]
        head, body = self._sections(text)[int(idx or 0)]
        return {'node_id': node_id, 'document': title, 'url': url, 'heading': head, 'text': body}

    def memory_topics(self, limit=12):
        "One cluster per remembered document, which is the honest clustering of a handful of pages."
        return [{'topic': title, 'members': [url], 'n': len(self._sections(text))}
                for title, url, text in list(self.remembered.values())[:int(limit)]]

    def memory_forget(self, doc_id):
        return self.remembered.pop(str(doc_id), None) is not None

    # -- the parts a terminal cannot have ------------------------------------
    @property
    def scopes(self):
        "Both, so the overlay branch of `inspect_python` has somewhere to be tested."
        return ('isolated', 'overlay')

    def inspect_python(self, code, scope='isolated'):
        """Isolated runs on a copy; overlay runs against the real namespace but refuses to move
        anything already in it -- the same guarantee an IDE's AST policy gives, enforced by
        restoring every pre-existing binding afterwards.
        """
        if scope != 'overlay': return super().inspect_python(code, scope)
        before = dict(self.ns)
        try: return self._exec(code, self.ns)
        except Exception as e: return f'{agent_err(e)}'
        finally:
            for k, v in before.items(): self.ns[k] = v
            for k in [k for k in self.ns if k not in before]: pass   # the agent's own names persist

    @property
    def kernel_kind(self): return 'ipymini'

    def note(self, text):
        self.notes.append(str(text))
        super().note(text)

Seeded with a file, a web of exactly one page, and a terminal that has already printed
something, it supports the entire tool surface -- twenty-four tools, against `LocalHost`'s
nineteen and `MemHost`'s twelve, and twenty-six once skills are wired in:

In [ ]:
host = FullHost(
    files={'pkg/sizes.py': 'RESERVE = 16_384\n\ndef threshold(ctx):\n    return max(1, ctx - min(RESERVE, ctx // 4))\n'},
    pages={'https://nbdev.fast.ai/api/export.html': (
        '# Exporting a notebook to a library\n\n'
        '## nb_export\n\n'
        'Named exports append to a module another notebook owns, so a second run duplicates them.\n\n'
        '## default_exp\n\n'
        'One module per notebook, named by the default_exp directive.\n')},
    terminal='$ pytest -q\n3 failed, 66 passed')
[t.__name__ for t in tools_for(host)]

['search_code',
 'similar_code',
 'outline',
 'list_files',
 'view_file',
 'edit_file',
 'create_file',
 'notebook_cells',
 'view_cell',
 'edit_cell',
 'add_cell',
 'web_search',
 'read_url',
 'research',
 'memory_search',
 'memory_tree',
 'memory_read',
 'memory_topics',
 'memory_forget',
 'list_vars',
 'run_python',
 'scale_numeric',
 'inspect_python',
 'read_terminal']

In [ ]:
# `FullHost` supports every group, including the shell: that is what makes it the double
# for a real application rather than for a bare test.
test_eq(len(tools_for(host)), 28)
test_eq('run_shell' in {t.__name__ for t in tools_for(host)}, True)
test_eq(set(t.__name__ for t in tools_for(NullHost())) < set(t.__name__ for t in tools_for(host)), True)
host.kernel_kind, host.concurrent, host.scopes

('ipymini', True, ('isolated', 'overlay'))

It is a real `LocalHost` underneath, so the sandbox, the file tools and the live namespace are
the production ones rather than stubs.

In [ ]:
view_file, replace_text, edit_file, create_file = file_tools(host)
print(view_file('pkg/sizes.py'))

1|d884|RESERVE = 16_384
2|0000|
3|b1c7|def threshold(ctx):
4|9825|    return max(1, ctx - min(RESERVE, ctx // 4))


In [ ]:
test_fail(lambda: host.check('/etc/passwd'), contains='outside the open folders')
host.run_python('total = threshold_calls = 0\ntotal'), host.list_vars().splitlines()

('0',
 ['total                int          0',
  'threshold_calls      int          0'])

The web is the pages it was given, and reading one remembers it -- which is what gives the
memory tools something to find.

In [ ]:
web_search, read_url, research = web_tools(host)
print(web_search('nbdev'))
print(read_url('https://nbdev.fast.ai/api/export.html')[:80])

Exporting a notebook to a library
  https://nbdev.fast.ai/api/export.html
# Exporting a notebook to a library

## nb_export

Named exports append to a mod


Remembered pages are recalled as document *sections* with breadcrumbs, not as flat snippets,
which is the distinction the memory tools exist to make.

In [ ]:
memory_search, memory_tree, memory_read, memory_topics, memory_forget = memory_tools(host)
host.memory_search('duplicates')

[{'node_id': 'doc_1#1',
  'document': 'Exporting a notebook to a library',
  'url': 'https://nbdev.fast.ai/api/export.html',
  'breadcrumb': 'Exporting a notebook to a library › nb_export',
  'text': 'Named exports append to a module another notebook owns, so a second run duplicates them.'}]

In [ ]:
test_eq(len(host.remembered), 1)
host.memory_tree()[0]['headings']

[{'node_id': 'doc_1#0', 'heading': 'Exporting a notebook to a library'},
 {'node_id': 'doc_1#1', 'heading': 'nb_export'},
 {'node_id': 'doc_1#2', 'heading': 'default_exp'}]

And forgetting is real: the document, its sections and everything derived from it go.

In [ ]:
doc_id = next(iter(host.remembered))
memory_forget(doc_id), host.memory_search('duplicates')

('forgot document', [])

`research` reads every matching page into one cited digest, so the slower path has somewhere
to be tested too.

In [ ]:
print(research('export')[:220])

# export

## Exporting a notebook to a library

# Exporting a notebook to a library

## nb_export

Named exports append to a module another notebook owns, so a second run duplicates them.

## default_exp

One module per 


The overlay scope runs the real interpreter against the real namespace and still cannot move
what is already there -- the guarantee an IDE enforces with an AST policy, enforced here by
restoring every pre-existing binding.

In [ ]:
host.run_python('kept = [1, 2, 3]')
host.inspect_python('kept.append(99)\nlen(kept)', scope='overlay'), host.run_python('len(kept)')

('4', '4')

The isolated scope cannot reach it at all: the append lands on the copy, and the real list is
the length it always was.

In [ ]:
host.inspect_python('kept.append(4)\nlen(kept)', scope='isolated'), host.run_python('len(kept)')

('5', '5')

The terminal is what the process printed, which is how `read_terminal` gets tested without an
IDE, and `note` is recorded rather than dropped.

In [ ]:
list_vars, run_python, scale_numeric, inspect_python, read_terminal = session_tools(host)
print(read_terminal())

$ pytest -q
3 failed, 66 passed


In [ ]:
host.note('compacted 12 messages')
host.notes

['compacted 12 messages']

## A backend with no model

`FakeBackend` answers from a list. It is a real `Backend` subclass, so the lock, the usage
accounting, the problem list and the history replacement are the production ones -- only
the generation is fiction.

In [ ]:
#| export
class FakeBackend(Backend):
    "A backend over a scripted list of replies, so a turn can be driven with no model at all."

    kind = 'fake'

    def __init__(self, spec=SPEC, replies=(), **kw):
        super().__init__(spec, **kw)
        self.replies, self.sent, self.hist_ = list(replies), [], []
        self.spawned = []

    def _start(self): return self
    def _close(self): pass

    def _send(self, msg, **kw):
        self.sent.append(msg)
        self.hist_.append({'role': 'user', 'content': str(msg)})
        out = self.replies.pop(0) if self.replies else '(done)'
        self.hist_.append({'role': 'assistant', 'content': out})
        return out

    def _stream(self, msg, **kw):
        for w in self._send(msg, **kw).split(' '): yield w + ' '

    def _oneshot(self, prompt, sp, max_tokens): return f'ONESHOT:{prompt[:40]}'
    def _usage(self):
        # Cumulative, the way a real chat's counters are: `Backend.send` assigns rather than
        # adds, so a double keeps its own running total or it cannot catch double-counting.
        n = max(1, len(self.sent))
        return Usage(model=self.spec.model_id, input=10*n, output=5*n, total=15*n, turns=n)

    @property
    def hist(self): return self.hist_

    def _replace_hist(self, summary, keep):
        self.hist_ = [{'role': 'user', 'content': summary}] + list(keep)

    def spawn(self, sp='', tools=(), **kw):
        s = FakeBackend(self.spec, replies=['sub answer'], sp=sp, tools=tools, shared=True)
        self.spawned.append(s)
        return s

In [ ]:
be = FakeBackend(replies=['the first answer', 'the second'])
be.send('a question'), be.send('another'), be.send('one more than were scripted')

('the first answer', 'the second', '(done)')

History accumulates in both roles, which is what compaction needs to have something to
compact.

In [ ]:
be.hist[:2]

[{'role': 'user', 'content': 'a question'},
 {'role': 'assistant', 'content': 'the first answer'}]

Streaming yields the same reply a word at a time, so a frontend's incremental rendering can
be driven without a model.

In [ ]:
''.join(FakeBackend(replies=['streamed one word at a time']).stream('go'))

'streamed one word at a time '

Usage is reported like a real turn's, so a status bar and a session total can be checked.

In [ ]:
be.use, be.use + be.use

(15 tok · in 10 · out 5 · model, 30 tok · in 20 · out 10 · model)

`fake_agent` wires one of these into a real `Agent`, with every job routed to it -- the
fastest way to drive a turn end to end.

In [ ]:
#| export
def fake_agent(host=None, replies=(), **kw):
    "An `Agent` whose every job routes to one `FakeBackend`. Returns `(agent, backend)`."
    a = Agent(host or MemHost({'/proj/a.py': 'def a(): pass\n'}), extensions=False, **kw)
    be = FakeBackend(SPEC, replies=replies)
    a._be = lambda job='turn': be
    a._be_or_none = lambda job='turn': be
    return a, be

#: What a local Gemma says when it refuses a turn. Real output, kept verbatim, because the
#: whole point of `native` is recognising the shape of this rather than a tidied version.

In [ ]:
agent, backend = fake_agent(replies=['`threshold` lives in `ramabana/runtime.py`.'])
agent.ask('where is the compaction threshold?')

'`threshold` lives in `ramabana/runtime.py`.'

It is a real agent: the tools are built, the activity feed recorded the preflight search,
and the usage was accounted.

In [ ]:
len(agent.tools), agent.turn_lines(), repr(agent.use)

## Compaction, end to end

With a fake backend and its 1000-token window, the whole compaction path runs: the threshold
is crossed, a summary is produced by the summary model, and the conversation is replaced by
that checkpoint plus the recent tail.

In [ ]:
for i in range(6): agent.ask(f'question {i} about the exporter')
len(backend.hist), agent.compactor.due(backend)

In [ ]:
summary = agent.compact()
agent.compactor.note

The history now begins with the checkpoint, and the model is told what happened to it --
that its context was rewritten while the kernel was not.

In [ ]:
print(backend.hist[0]['content'][:200])

In [ ]:
test_eq(backend.hist[0]['content'].startswith('Previous conversation summary:'), True)
test_eq(agent.compactor.count, 1)
agent.compactor.last[:60]

## A backend that mutters

The failure this harness exists for: a local engine that prints its complaint to a file
descriptor, returns an empty string, and raises nothing. `GEMMA` is real output, kept
verbatim, because the point is recognising the shape of the real thing rather than a tidied
version of it.

In [ ]:
#| export
GEMMA = ('E0000 00:00:1234.567 llm_engine.cc:412] input token IDs exceed the maximum '
         'number of tokens 4096, got 5092\n')


class MutteringBackend(Backend):
    "A backend that fails the way litert does: it prints, returns nothing, and raises nothing."

    kind = 'muttering'

    def __init__(self, spec=None, mode='empty', **kw):
        super().__init__(spec or ModelSpec('gemma-e2b', 'muttering', 'gemma/e2b', ctx=4096), **kw)
        self.mode = mode

    def _start(self): return self
    def _close(self): pass
    def _usage(self): return Usage(model=self.spec.model_id)

    def _mutter(self):
        "What `RishiBackend._native` does for real: keep what the engine said, and report it."
        self.last_native = GEMMA.strip()
        self.problem(f'{self.spec.name}: {GEMMA.strip()}')

    def _send(self, msg, **kw):
        self._mutter()
        return ''

    def _stream(self, msg, **kw):
        self._mutter()
        return iter(())

    def _oneshot(self, prompt, sp, max_tokens):
        self._mutter()
        raise RuntimeError('input too long')

In [ ]:
mute = MutteringBackend()
mute.send('anything'), mute.problems

That is the whole contract: the caller gets a sentence that is true instead of an empty
string, and the engine's own words are kept for the status line.

In [ ]:
test_eq(mute.last_native, GEMMA.strip())
mute.send('again')

In [ ]:
test_eq(len(mute.problems), 1)            # the same complaint twice is one problem
mute.spec.ctx

## A backend that performs

`ScriptedBackend` plays a list of `Step`s and streams its words one at a time. `token_delay`
is what makes a screenshot possible: at zero the turn finishes before the first repaint, and
every capture shows a completed answer -- the one thing a streaming screenshot must not
show. Its tool steps go through the agent's *real* tool list, so the activity feed, the
approval gate and the file snapshots all run for real.

In [ ]:
#| export
class Step:
    """One thing a scripted model does: call a tool, or say something.

    `tool` is a `(name, kwargs)` pair and is called through the agent's real tool list, so
    the activity feed, the approval gate and the file snapshots all run for real -- the only
    fiction is which tool the model decided to call.
    """

    def __init__(self, text='', tool=None, pause=0.0):
        self.text, self.tool, self.pause = text, tool, pause


class ScriptedBackend(Backend):
    """Plays a list of `Step`s, streaming its words one at a time.

    `token_delay` is what makes a screenshot possible: at zero the turn finishes before the
    first repaint and every capture shows a completed answer, which is the one thing a
    streaming screenshot must not show.
    """

    kind = 'scripted'

    #: What a spawned sub-agent answers, keyed by a substring of the question. A fan-out
    #: whose three sub-agents all say the same thing would look like one call in a wig.
    SUB_ANSWERS = {'import': 'three files: backend.py, models.py, fastllm_hitl.py',
                   'compaction': 'chat.py:compact(), fired from _prepare() at the threshold',
                   'shape': 'df is (200, 2); `keep` is an int'}

    def __init__(self, spec=SCRIPTED, steps=(), token_delay=0.02, **kw):
        super().__init__(spec, **kw)
        self.steps, self.token_delay = list(steps), token_delay
        self.hist_, self.sent = [], []

    def _start(self): return self
    def _close(self): pass
    def _oneshot(self, prompt, sp, max_tokens): return 'a one-shot answer'
    def _usage(self): return Usage(model=self.spec.model_id, input=1840, output=96, total=1936,
                                   cached=1200, cost=0.0031, turns=1)

    @property
    def hist(self): return self.hist_

    def _replace_hist(self, summary, keep):
        self.hist_ = [{'role': 'user', 'content': summary}] + list(keep)

    def spawn(self, sp='', tools=(), **kw):
        answers = self.SUB_ANSWERS
        class Sub(ScriptedBackend):
            def _run(self, msg):
                q = str(msg).lower()
                hit = next((v for k, v in answers.items() if k in q), 'nothing found')
                for w in hit.split(' '): yield w + ' '
        return Sub(self.spec, token_delay=0, shared=True)

    def _tool(self, name):
        return next((t for t in self.tools if getattr(t, '__name__', '') == name), None)

    def _run(self, msg):
        "Walk the script, calling tools for real and yielding the words of every text step."
        self.sent.append(msg)
        self.hist_.append({'role': 'user', 'content': str(msg)})
        for s in self.steps:
            if s.pause: time.sleep(s.pause)
            if s.tool:
                name, kw = s.tool
                if (f := self._tool(name)) is not None:
                    self.hist_.append({'role': 'assistant', 'content': f'[{name}]'})
                    self.hist_.append({'role': 'tool', 'content': str(f(**kw))[:400]})
            for w in s.text.split(' '):
                if not w: continue
                if self.token_delay: time.sleep(self.token_delay)
                yield w + ' '

    def _send(self, msg, **kw):
        out = ''.join(self._run(msg))
        self.hist_.append({'role': 'assistant', 'content': out})
        return out

    def _stream(self, msg, **kw):
        out = []
        for w in self._run(msg):
            out.append(w)
            yield w
        self.hist_.append({'role': 'assistant', 'content': ''.join(out)})

In [ ]:
script = [Step('Looking at the file.', tool=('view_file', {'path': '/proj/a.py'})),
          Step('It returns 1, so the caller gets an int.')]
play = ScriptedBackend(steps=script, token_delay=0)
play.refresh('', [t for t in agent.tools if t.__name__ == 'view_file'])
play.send('what does a() return?')

The tool really ran, and its result is in the history where a compaction or a UI would find
it.

In [ ]:
[m['role'] for m in play.hist], play.hist[2]['content'][:40]

A spawned sub-agent answers differently per question, because a fan-out whose three
sub-agents all say the same thing would look like one call in a wig.

In [ ]:
sub = play.spawn()
''.join(sub._run('which files import fastllm?')), ''.join(sub._run('what shape is df?'))

In [ ]:
test_eq(play.use.total, 1936)
play.use

## A real model with every tool

The doubles above prove control flow. This final check proves the assembled product: an
actual MLX model loaded through rishi, a `FullHost` with every capability, and all twenty-eight
tools installed in the model conversation. It is `eval: false` so CI needs neither Apple
silicon nor a multi-gigabyte cache; the outputs stored below are from the real run.

In [ ]:
#| eval: false
register_model('release-model', 'mlx-community/Ornith-1.0-9B-8bit', runtime='mlx', ctx=32_768)
real_host = FullHost(
    files={'pkg/sizes.py': 'RESERVE = 16_384\n\ndef threshold(ctx):\n    return ctx - RESERVE\n'},
    pages={'https://example.test/nbdev': '# nbdev export\n\nOne module belongs to one notebook.'},
    terminal='$ pytest -q\n69 passed')
real_agent = Agent(real_host, model='release-model', extensions=False)
for job in ('classify', 'summary', 'completion', 'subagent'):
    real_agent.routing.set('release-model', job)
real_agent.start() is not None, real_agent.note

/Users/71293/code/personal/orgs/ramabana/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(True, 'release-model · local · 32k ctx · 28 tools')

`Agent.tools` is the complete surface: code, files, notebooks, web, memory, live session,
skills, and both delegation tools. No group is represented by a stub that always raises.

In [ ]:
#| eval: false
real_tool_names = [tool.__name__ for tool in real_agent.tools]
real_tool_names

['search_code',
 'similar_code',
 'outline',
 'list_files',
 'view_file',
 'edit_file',
 'create_file',
 'notebook_cells',
 'view_cell',
 'edit_cell',
 'add_cell',
 'web_search',
 'read_url',
 'research',
 'memory_search',
 'memory_tree',
 'memory_read',
 'memory_topics',
 'memory_forget',
 'list_vars',
 'run_python',
 'scale_numeric',
 'inspect_python',
 'read_terminal',
 'read_skill',
 'create_skill',
 'delegate_search',
 'delegate_parallel']

In [ ]:
#| eval: false
expected = {
    'search_code', 'similar_code', 'outline', 'list_files',
    'view_file', 'edit_file', 'create_file',
    'notebook_cells', 'view_cell', 'edit_cell', 'add_cell',
    'web_search', 'read_url', 'research',
    'memory_search', 'memory_tree', 'memory_read', 'memory_topics', 'memory_forget',
    'list_vars', 'run_python', 'scale_numeric', 'inspect_python', 'read_terminal',
    'read_skill', 'create_skill', 'delegate_search', 'delegate_parallel'}
test_eq(set(real_tool_names), expected)
len(real_tool_names)

28

The list is not merely attached to `Agent`: rishi's live conversation compiled all
twenty-eight schemas. This is the boundary that catches a missing annotation, a lost
`functools.wraps`, or a tool shape the runtime cannot accept.

In [ ]:
#| eval: false
real_backend = real_agent.backend
test_eq(real_backend.ready, True)
test_eq(len(real_backend.chat.tools), 28)
test_eq(len(real_backend.chat.toolspecs), 28)
(real_backend.spec.model_id, len(real_backend.chat.toolspecs))

('mlx-community/Ornith-1.0-9B-8bit', 28)

And the loaded model generates. The probe is routed to that same engine and asserted
non-empty, so this cannot pass merely because model construction accepted the tool schemas.

In [ ]:
#| eval: false
real_reply = real_agent.oneshot(
    'Reply with exactly READY and nothing else.',
    'Return only the requested answer.', job='classify', max_tokens=192)
assert real_reply.strip()
real_reply

'READY'

In [ ]:
#| eval: false
real_agent.close()
real_agent.status()['model'], real_agent.use.total

('release-model', 0)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()